In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Keys you still need
for key in ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]:
    print(key, "OK" if os.getenv(key) else "MISSING")

TAVILY_API_KEY OK
LANGSMITH_API_KEY MISSING


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",            # use the exact name from `ollama list`
    temperature=0,
    reasoning=False,             # turn off Qwen3's thinking mode
    validate_model_on_init=True, # error right away if the model name is wrong
)

print(llm.invoke("Reply with just the word OK").content)

/Users/sportssciencecompany/Desktop/RAG/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


OK


In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

urls = [
    # Anthropic engineering: agents, context, tools, MCP
    "https://www.anthropic.com/engineering/building-effective-agents",                  # Dec 2024
    "https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents", # Sep 2025
    "https://www.anthropic.com/engineering/multi-agent-research-system",                # Jun 2025
    "https://www.anthropic.com/engineering/code-execution-with-mcp",                    # Nov 2025
    "https://www.anthropic.com/engineering/writing-tools-for-agents",                   # 2025 (unverified)
    "https://www.anthropic.com/news/contextual-retrieval",                              # Sep 2024, RAG technique
    # Lilian Weng: reasoning, reward hacking, hallucination
    "https://lilianweng.github.io/posts/2025-05-01-thinking/",                          # May 2025
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",                    # Nov 2024
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",                     # Jul 2024
    # Evals
    "https://hamel.dev/blog/posts/evals-faq/",                                          # expanded Sep 2026
    "https://hamel.dev/blog/posts/evals/",                                              # 2024 classic
    # Agents
    "https://huyenchip.com/2025/01/07/agents.html",                                     # Jan 2025 (unverified)
]

# Load each URL separately so one bad URL doesn't stop the whole cell
docs_list = []
for url in urls:
    try:
        loaded = WebBaseLoader(url).load()
        chars = sum(len(d.page_content) for d in loaded)
        print(f"{chars:>8} chars  {url}")
        if chars < 2000:
            print("          ^ suspiciously short, skipping (blocked page or wrong URL?)")
            continue
        docs_list.extend(loaded)
    except Exception as e:
        print(f"  FAILED  {url} -> {type(e).__name__}: {e}")

print(f"\nLoaded {len(docs_list)} documents")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"Split into {len(doc_splits)} chunks")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)
vectorstore.reset_collection()

batch_size = 64
for i in range(0, len(doc_splits), batch_size):
    vectorstore.add_documents(doc_splits[i : i + batch_size])
    print(f"Indexed {min(i + batch_size, len(doc_splits))}/{len(doc_splits)}")

retriever = vectorstore.as_retriever()
print("Total in store:", vectorstore._collection.count())

   18816 chars  https://www.anthropic.com/engineering/building-effective-agents
   21671 chars  https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
   26599 chars  https://www.anthropic.com/engineering/multi-agent-research-system
   14585 chars  https://www.anthropic.com/engineering/code-execution-with-mcp
   22100 chars  https://www.anthropic.com/engineering/writing-tools-for-agents
   15777 chars  https://www.anthropic.com/news/contextual-retrieval
   56363 chars  https://lilianweng.github.io/posts/2025-05-01-thinking/
   51357 chars  https://lilianweng.github.io/posts/2024-11-28-reward-hacking/
   42734 chars  https://lilianweng.github.io/posts/2024-07-07-hallucination/
  119257 chars  https://hamel.dev/blog/posts/evals-faq/
   24906 chars  https://hamel.dev/blog/posts/evals/
   50907 chars  https://huyenchip.com/2025/01/07/agents.html

Loaded 12 documents
Split into 547 chunks
Indexed 64/547
Indexed 128/547
Indexed 192/547
Indexed 256/547
Indexed 320/5

In [6]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
print(len(embeddings.embed_query("hello world")))   # expect 768

768


In [7]:
lengths = sorted((len(d.page_content), i) for i, d in enumerate(doc_splits))
print("chunks:", len(doc_splits))
print("longest 5 (chars, index):", lengths[-5:])

chunks: 547
longest 5 (chars, index): [(1429, 10), (1455, 48), (1480, 49), (1483, 18), (1613, 50)]


In [9]:
import os
from typing import Literal

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

LOCAL_LLM = os.getenv("LOCAL_LLM", "qwen3:8b")  # add LOCAL_LLM=qwen3:8b to .env


class GradeDocument(BaseModel):
    """Relevance check for a retrieved document."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to the question, otherwise 'no'"
    )


llm = ChatOllama(model=LOCAL_LLM, temperature=0, reasoning=False)
structured_grader_llm = llm.with_structured_output(GradeDocument, method="json_schema")

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing whether a retrieved document is relevant to a user question. "
     "If the document contains information or keywords related to the question, grade it as relevant. "
     "It does not need to be a strict test; the goal is to filter out clearly wrong retrievals."),
    ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
])

retrieval_grader = grade_prompt | structured_grader_llm

In [11]:
question = "How do AI agents manage memory and context?"
docs = retriever.invoke(question)

for d in docs:
    result = retrieval_grader.invoke({"question": question, "document": d.page_content})
    print(result.binary_score, "|", d.metadata["source"], "|", d.page_content[:80].replace("\n", " "))

# Negative test: an unrelated question should get "no"


yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | components of context (system prompts, tools, examples, message history, etc) is
yes | https://huyenchip.com/2025/01/07/agents.html | This post covered conceptually how agents work and different components of an ag
yes | https://www.anthropic.com/news/contextual-retrieval | Contextual retrieval in AI systems \ AnthropicSkip to main contentNewsResearchPo
yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | history, etc).An agent running in a loop generates more and more data that could


In [ ]:
off_topic = retrieval_grader.invoke({"question": "What is a good biryani recipe?",
                                     "document": docs[0].page_content})
print("is the question relevant?", off_topic.binary_score)

off-topic -> no
